In [1]:
import pandas as pd

# load the sample dataset
df = pd.read_csv("0.5_cdc_case_surveillance_sample.csv")

In [2]:
print("Initial Shape:", df.shape)
print("\nInitial Columns:\n", df.columns.tolist())
print("\nInitial Missing Values:\n", df.isna().sum())
print("\nInitial Data Types:\n", df.dtypes)

Initial Shape: (191662, 12)

Initial Columns:
 ['cdc_case_earliest_dt', 'cdc_report_dt', 'pos_spec_dt', 'onset_dt', 'current_status', 'sex', 'age_group', 'race_ethnicity_combined', 'hosp_yn', 'icu_yn', 'death_yn', 'medcond_yn']

Initial Missing Values:
 cdc_case_earliest_dt            0
cdc_report_dt               14886
pos_spec_dt                108631
onset_dt                   121492
current_status                  0
sex                             0
age_group                       0
race_ethnicity_combined         0
hosp_yn                         0
icu_yn                          0
death_yn                        0
medcond_yn                      0
dtype: int64

Initial Data Types:
 cdc_case_earliest_dt       object
cdc_report_dt              object
pos_spec_dt                object
onset_dt                   object
current_status             object
sex                        object
age_group                  object
race_ethnicity_combined    object
hosp_yn                    obje

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Cleaned Column Names:\n", df.columns.tolist())

Cleaned Column Names:
 ['cdc_case_earliest_dt', 'cdc_report_dt', 'pos_spec_dt', 'onset_dt', 'current_status', 'sex', 'age_group', 'race_ethnicity_combined', 'hosp_yn', 'icu_yn', 'death_yn', 'medcond_yn']


In [4]:
df = df.replace(["Missing", "missing", ""], pd.NA)

In [5]:
date_cols = ["cdc_case_earliest_dt", "cdc_report_dt", "pos_spec_dt", "onset_dt"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [6]:
df = df.dropna(subset=["cdc_case_earliest_dt"])

In [7]:
categorical_cols = [
    "current_status",
    "sex",
    "age_group",
    "race_ethnicity_combined"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

In [8]:
binary_cols = ["hosp_yn", "icu_yn", "death_yn", "medcond_yn"]

binary_map = {
    "Yes": 1,
    "No": 0
}

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map(binary_map).fillna(-1).astype("int8")

In [9]:
df["year"] = df["cdc_case_earliest_dt"].dt.year
df["month"] = df["cdc_case_earliest_dt"].dt.month
df["day_of_week"] = df["cdc_case_earliest_dt"].dt.day_name()

In [10]:
print("Missing in onset_dt:", df["onset_dt"].isna().sum())
print("Percent missing in onset_dt:", round(df["onset_dt"].isna().mean() * 100, 2), "%")

Missing in onset_dt: 121492
Percent missing in onset_dt: 63.39 %


In [11]:
df = df.drop(columns=["onset_dt"])

Dropped due to a high proportion of missing values and its redundancy with the primary case date (cdc_case_earliest_dt). Retaining this column would introduce noise without adding much meaningful analytical value.

In [12]:
print("Final Shape:", df.shape)
print("\nMissing Values After Cleaning:\n", df.isna().sum())
print("\nData Types After Cleaning:\n", df.dtypes)

Final Shape: (191662, 14)

Missing Values After Cleaning:
 cdc_case_earliest_dt            0
cdc_report_dt               14886
pos_spec_dt                108631
current_status                  0
sex                             0
age_group                       0
race_ethnicity_combined         0
hosp_yn                         0
icu_yn                          0
death_yn                        0
medcond_yn                      0
year                            0
month                           0
day_of_week                     0
dtype: int64

Data Types After Cleaning:
 cdc_case_earliest_dt       datetime64[ns]
cdc_report_dt              datetime64[ns]
pos_spec_dt                datetime64[ns]
current_status                     object
sex                                object
age_group                          object
race_ethnicity_combined            object
hosp_yn                              int8
icu_yn                               int8
death_yn                             int8
med

cdc_report_dt and pos_spec_dt were kept even though they have some missing values because they capture different parts of the timeline (reporting vs testing). Even if they’re not complete, they can still be useful later for looking at delays or how cases progress over time.

In [13]:
df.to_csv("0.5_cleaned_sample_dataset.csv", index=False)
print("Cleaned dataset saved as cleaned_sample_dataset.csv")

Cleaned dataset saved as cleaned_sample_dataset.csv
